In [0]:
-- Department Preferences by Customer Segment
WITH customer_metrics AS (
  SELECT
    o.user_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(op.reordered) * 100.0 / COUNT(*), 2) AS reorder_rate_pct
  FROM instacart_gold.orders o
  INNER JOIN instacart_gold.order_products op ON o.order_id = op.order_id
  GROUP BY o.user_id
),
customer_segments AS (
  SELECT
    user_id,
    CASE
      WHEN total_orders >= 50 AND reorder_rate_pct >= 60 THEN 'VIP Loyalist'
      WHEN total_orders >= 30 AND reorder_rate_pct >= 50 THEN 'Loyal Regular'
      WHEN total_orders >= 15 THEN 'Regular Customer'
      WHEN total_orders >= 5 THEN 'Occasional Shopper'
      ELSE 'New Customer'
    END AS customer_segment
  FROM customer_metrics
)
SELECT
  cs.customer_segment,
  p.department,
  COUNT(op.product_id) AS products_purchased,
  ROUND(COUNT(op.product_id) * 100.0 / SUM(COUNT(op.product_id)) OVER (PARTITION BY cs.customer_segment), 2) AS pct_of_segment_purchases
FROM instacart_gold.orders o
INNER JOIN customer_segments cs ON o.user_id = cs.user_id
INNER JOIN instacart_gold.order_products op ON o.order_id = op.order_id
INNER JOIN instacart_gold.products p ON op.product_id = p.product_id
GROUP BY cs.customer_segment, p.department
QUALIFY ROW_NUMBER() OVER (PARTITION BY cs.customer_segment ORDER BY products_purchased DESC) <= 5
ORDER BY cs.customer_segment, products_purchased DESC;